In [1]:
import datetime

In [2]:
import fsspec
import xarray as xr
import scipy.spatial
import numpy as np
import os
import argparse
from datetime import date
from datetime import datetime as dt
from calculations.calculations import vapor_pressure
from calculations.calculations import wind_tot
from calculations.calculations import rel_hum
import regionmask
import geopandas as gpd

### Main
cleaned up version of extract_vars_from_ERA5 from Maile's code. 

reasoning for a temp netcdf file to save the extracted variables: 
- it takes a long time to access the Google cloud ERA5 files
- CPU times: user 5.09 s, sys: 701 ms, total: 5.79 s
Wall time: 60 s
(but vs save data..?)

If climate metric calculate and statistics code already exist, can do online extract (e.g. for time period update)
But if say developing climate metric calculation, should do file download to local, but saving extracted ERA5 files take quite a while. (35 min for 2 variables)

Q: potential reasoning for often accessing data - do they want to manually update the "10-year stat" window - e.g. year 2026 - want the last 10 years (2016-2025); 2017-2026 in 2027? Basically rolling year vs rolling month (year 2028 july want (2017july thru 2028 June) ? 

Updated on Feb17 2026 to extract less grid points (variables prior this date will have grid points 29x27, onward will only contain 22x19)

In [4]:
%%time 
fs = fsspec.filesystem('gs')
fs.ls('gs://gcp-public-data-arco-era5/co/')

# Opening dataset with zarr
reanalysis = xr.open_zarr(
    'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3', 
    chunks={'time': 48},
    consolidated=True,
    )

CPU times: user 2.82 s, sys: 209 ms, total: 3.03 s
Wall time: 15.4 s


In [5]:
# Define vars to be called 
variable = ['2m_temperature','10m_u_component_of_wind','10m_v_component_of_wind','2m_dewpoint_temperature','total_precipitation'] #EDIT /data/keeling/a/rytam2/a/iema_output/variables_202510210102.nc
# calc = #EDIT 
year_start = 2016 #EDIT 
year_end = 2025 #EDIT 

# Define Once
# Dates
i_date = str(year_start) + '-01-01'
f_date = str(year_end)   + '-12-31'


recent_an = reanalysis.sel(time=slice(i_date, f_date))
era5_var = recent_an[variable]

#lon bounds of IL
lon1 = -91.75
lon2 = -86.75 

lon_min = (lon1%360)
lon_max = (lon2%360)
lat_min = 37.0
lat_max = 42.75

illinois_ds = era5_var.where(
(recent_an.longitude > lon_min) & (recent_an.latitude > lat_min) &
(recent_an.longitude < lon_max) & (recent_an.latitude < lat_max),
drop=True).rename({'longitude':'lon', 'latitude':'lat'})


fin_array = illinois_ds

In [6]:
t2m = fin_array[variable[0]]
skt = fin_array[variable[1]]

In [7]:
t2m

<xarray.DataArray '2m_temperature' (time: 87672, lat: 22, lon: 19)> Size: 147MB
dask.array<where, shape=(87672, 22, 19), dtype=float32, chunksize=(48, 22, 19), chunktype=numpy.ndarray>
Coordinates:
  * lat      (lat) float32 88B 42.5 42.25 42.0 41.75 ... 38.0 37.75 37.5 37.25
  * lon      (lon) float32 76B 268.5 268.8 269.0 269.2 ... 272.5 272.8 273.0
  * time     (time) datetime64[ns] 701kB 2016-01-01 ... 2025-12-31T23:00:00
Attributes:
    long_name:   2 metre temperature
    short_name:  t2m
    units:       K

### Save

In [ ]:
ds = xr.Dataset({'t2m':t2m, 'skt':skt})
ds

In [23]:
%%time
filename = '/data/keeling/a/rytam2/a/iema_output/era5/era5_extract/variables_'+dt.now().strftime("%Y%m%d%H%M")+'.nc'
ds.to_netcdf(filename)
print(filename)

/data/keeling/a/rytam2/a/iema_output/variables_202602080250.nc
CPU times: user 28min 4s, sys: 7min 41s, total: 35min 45s
Wall time: 29min 53s
